In [69]:
from costs_concrete import *
from math import comb

# Calculating BIM server storage with rebalancing

The below function takes as input n (DB size in records) and CCmax (an upper bound on the download), and finds the smallest possible storage (in records) that can be obtained using the BIM construction together with rebalancing.

It does this by searching over values of m and D and determining the number of records r that need to be combined into one "mega-record" for rebalancing.

In [70]:
# n: db size in records
# CCmax: upper bound on download in records
def get_bim_costs(n, CCmax):
    best_storage = float('inf')
    for m in range(2, maxm):
        for D in range(1, min(m+1, maxD)):
            # number of mega-records
            n2 = db_size(D, 1, m, 2, homog=True)

            # number of records per mega-record
            r = int(np.ceil(n/n2))

            # download in records
            CC = r * server_online_time(D, 1, m, 2, 2, homog=True, use_finite_differences=False)

            if CC <= CCmax:
                test_storage = r * preprocessing_cost(D, 1, m, 2, 2, homog=True, use_finite_differences=False)

                if test_storage < best_storage:
                    best_storage = test_storage
                    param_dict = {'m': m, 'D': D, 'r': r}

    return best_storage, param_dict

# Generating table 1

The below example is for the bottom row of the right subtable of table 1

In [147]:
D = 9
m = 35
l = 32 # length of a record in bytes

DB size, in GB

In [148]:
n = db_size(D, 1, m, 2, homog=True) * l
assert n == comb(m, D) * l
print(n/pow(1024, 3)) # GB

2.1042662858963013


Our server storage. For all examples in our table, this is 1 TB.

In [149]:
our_storage = preprocessing_cost(D, 1, m, 2, 2, homog=True, use_finite_differences=True) * l
assert our_storage == pow(2, m) * l
print(our_storage/pow(1024, 4)) # TB

1.0


In [150]:
our_CC = server_online_time(D, 1, m, 2, 2, use_finite_differences=True) * l
print(our_CC/pow(1024, 2)) # MB

1.81689453125


Calculating BIM server storage with rebalancing, in TB

In [151]:
tmp_storage, params = get_bim_costs(n/l, our_CC/l)
bim_storage = tmp_storage * l
params

{'m': 35, 'D': 9, 'r': 1}

In [152]:
blowup = bim_storage/our_storage

In [154]:
blowup/1e4

5.9536

BIM server storage if we didn't consider rebalancing

In [155]:
preprocessing_cost(D, 1, m, 2, 2, use_finite_differences=False) * l/our_storage

59536.0